In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

from agents.classifying_agent import ClassifyingAgent
import chromadb
from agents.agent import Agent

In [2]:
DB = "products_vectorstore"
client = chromadb.PersistentClient(path=DB)
collections = client.get_or_create_collection('products')

In [3]:
classifyingAgent = ClassifyingAgent(collection=collections)


# def classifyingAgent(product_description):
#     # agent = Agent(name="classifying_agent", description="Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.", tools=[{"type": "function", "function": classifyingAgent}])
#     # response = agent.run(product_description=product_description)
#     # return response
#     return "baby products, toys, home goods"

categorization_agent = {
    "name": "categorization_agent",
    "description": "Categorizes a product based on its description. If the product is difficult to categorize, the agent will return the 3 categories you think it could belong to.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_description": {
                "type": "string",
                "description": "A description of the product that needs to be categorized."
            },
        },
        "required": ["product_description"],
        "additionalProperties": False
    }
}

Using device: mps


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()


OpenAI API Key exists and begins sk-proj-


In [5]:


tools = [{"type": "function", "function": categorization_agent}]

def handle_tool_call(message):
    mapping = {
        "categorization_agent": classifyingAgent.classify
    }

    results = []

    for tool_call in message.tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        tool = mapping.get(tool_name)

        if tool:
            result = tool(**arguments)
        else:
            result = f"Unknown tool: {tool_name}"

        results.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })

    return results


In [6]:
WELCOME_MESSAGE = "Hi! My name is Peeta, and I'm here to help you register your product. First, please provide me with a description of your product. For example, what does it do, what are its features, and any other relevant information. Based on your description, I will sort your product into the correct category."
system_message ="""
You are a helpful assistant for an e-commerce platform that helps bussinesses register their products. Do not make up anything if you are unsure.
"""

In [ ]:

# from langchain_core import messages


chatbot = gr.Chatbot(
    type="messages",
    value=[
        {"role": "assistant", "content": WELCOME_MESSAGE}
    ]
)

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        print("this is smessage", message)
        response = handle_tool_call(message)
        print("this is response", response)
        # messages.append(message)
        # # messages.append(message.model_dump(exclude_none=True))
        # messages.append(response)
        messages.append(message.model_dump(exclude_none=True))
        messages.extend(response)
        # print("this is messages after appending", messages)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
        # print("this is final response", response)
    
    return response.choices[0].message.content

gr.ChatInterface(fn=chat,chatbot=chatbot ,type="messages").launch()

/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_35404/75278224.py:4: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


this is smessage ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_U8EbCWT7LTMNlmAHPQ5PU5J3', function=Function(arguments='{"product_description":"Sparking water with 30gr of protein in each can"}', name='categorization_agent'), type='function')])
this is the final response from the classify function Grocery & Gourmet Food
this is response [{'role': 'tool', 'content': 'Grocery & Gourmet Food', 'tool_call_id': 'call_U8EbCWT7LTMNlmAHPQ5PU5J3'}]
this is messages after appending [{'role': 'system', 'content': '\nYou are a helpful assistant for an e-commerce platform that helps bussinesses register their products. Do not make up anything if you are unsure.\n'}, {'role': 'assistant', 'content': "Hi! My name is Peeta, and I'm here to help you register your product. First, please provide me with a description of your product. For example, what does it do, what are its fe